(referencia1)=
# Programación asíncrona con asyncio en Python: una guía práctica

## ¿Por qué importa la asincronía?

Python ha madurado enormemente en los últimos años, y una de las áreas donde más ha crecido es la programación asíncrona. La biblioteca asyncio hace posible ejecutar múltiples tareas de forma concurrente dentro de un programa que, en esencia, corre en un solo hilo.

Cuando construyes aplicaciones que dependen de operaciones lentas —consultar una API externa, leer de una base de datos, procesar archivos grandes— el tiempo de espera se convierte en tu enemigo. asyncio te permite aprovechar ese tiempo de espera para seguir haciendo trabajo útil en lugar de quedarte bloqueado.

Este artículo te llevará por los pilares fundamentales de asyncio: corrutinas, gestores de contexto asíncronos, tareas concurrentes, medición de rendimiento, manejo de errores y el bucle de eventos. Cada sección incluye ejemplos listos para ejecutar.

## Corrutinas: la base de todo

Una corrutina es una función especial que puede suspender su ejecución en un punto determinado y ceder el control a otra tarea, para luego retomar desde donde se quedó. Esto es exactamente lo que necesitas cuando tienes operaciones que implican esperas.

En Python, una corrutina se declara con `async def`:

In [ ]:
async def saludar():
    print("Hola desde una corrutina")

Para ejecutarla, necesitas la palabra clave `await`, ya que no puedes invocarla como una función ordinaria. El punto de entrada de todo programa asyncio es `asyncio.run()`:

In [2]:
import asyncio

async def saludar():
    await asyncio.sleep(2)  # simula una operación de 2 segundos
    print("Operación completada")

async def main():
    await saludar()

if __name__ == "__main__":
    asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

NOTA: El código anterior nos genera un error porque se jecuta en jupyter notebook. El error ocurre porque `asyncio.run()` no se puede llamar cuando ya hay un event loop en ejecución, como en Jupyter, VS Code o entornos interactivos.

Se debe usar  `nest_asyncio` para permitir event loops anidados, ideal para un flujo de trabajo con notebooks. El código anterior corregido para poder ser ejecutado de manera adecuada con jupyter notebook, sería el siguiente:

In [3]:
import nest_asyncio
nest_asyncio.apply()  # Aplícalo una vez por sesión

import asyncio

async def saludar():
    await asyncio.sleep(2)
    print("Operación completada")

async def main():
    await saludar()

asyncio.run(main())

Operación completada


## Gestores de contexto e iteradores asíncronos

En aplicaciones reales, necesitas abrir conexiones, leer archivos o interactuar con servicios externos. asyncio ofrece soporte nativo para hacer todo esto de forma no bloqueante.

`async with`

El bloque async with gestiona recursos asíncronos de forma segura, garantizando que se abran y cierren correctamente aunque ocurra un error:

In [5]:
import aiofiles
import asyncio

async def procesar_log():
    async with aiofiles.open('registro_eventos.txt', mode='rb') as archivo:
        contenido_binario = await archivo.read()

    texto = contenido_binario.decode('utf-8')
    print(texto[:500])  # muestra los primeros 500 caracteres

async def main():
    await procesar_log()

if __name__ == "__main__":
    asyncio.run(main())

El registro de eventos es un sistema esencial para documentar incidencias en software de facturación. Captura automáticamente interacciones críticas con el SIF (Sistema Informático de Facturación), como cambios de configuración, fallos técnicos o accesos no autorizados, garantizando trazabilidad e integridad. Cada evento se firma electrónicamente y se almacena con timestamp preciso, permitiendo auditorías por la AEAT. Incluye resúmenes periódicos cada 6 horas y exportación de datos. Esta normati


d:\MisTrabajos\IA_generativa\Langchain\libro\.venv\Lib\site-packages\aiofiles\tempfile\__init__.py:11: RuntimeWarning: coroutine 'main' was never awaited
  from ..base import AiofilesContextManager


Otro caso muy común es hacer peticiones HTTP sin bloquear el hilo principal:

In [6]:
# Patrón habitual con aiohttp
async with aiohttp.ClientSession() as sesion:
    async with sesion.get('https://api.misitio.com/usuarios') as respuesta:
        datos = await respuesta.json()

NameError: name 'aiohttp' is not defined

**Buena práctica**: usa siempre async with para gestionar recursos que requieran inicialización y limpieza.

`async for`

Cuando necesitas recorrer datos que se generan de forma asíncrona, usas async for junto con un generador asíncrono:

In [7]:
import asyncio

async def generar_reportes():
    secciones = ["resumen", "detalle", "gráficas", "conclusiones"]
    for seccion in secciones:
        await asyncio.sleep(1)  # simula tiempo de generación
        yield f"Sección generada: {seccion}"

async def exportar_documento():
    async for bloque in generar_reportes():
        print(f"Procesando → {bloque}")

asyncio.run(exportar_documento())

Procesando → Sección generada: resumen
Procesando → Sección generada: detalle
Procesando → Sección generada: gráficas
Procesando → Sección generada: conclusiones


La diferencia entre for y async for es clara: el primero recorre iterables síncronos (listas, tuplas, generadores normales), mientras que el segundo está diseñado para iterables asíncronos. Intentar mezclarlos incorrectamente provoca un TypeError en tiempo de ejecución.

## Tareas y ejecución concurrente.

Cuando envuelves una corrutina en una `Task` usando `asyncio.create_task()`, la estás registrando para que el bucle de eventos la ejecute tan pronto como pueda, sin esperar a que la corrutina anterior termine:

In [8]:
import asyncio

async def descargar_imagen(nombre):
    print(f"Iniciando descarga: {nombre}")
    await asyncio.sleep(2)
    print(f"Descarga completada: {nombre}")

async def main():
    tarea = asyncio.create_task(descargar_imagen("foto_perfil.jpg"))
    await tarea

asyncio.run(main())

Iniciando descarga: foto_perfil.jpg
Descarga completada: foto_perfil.jpg


Si intentas crear una tarea fuera de un contexto asíncrono activo, Python lanzará:

````python
RuntimeError: no running event loop
````

Esto sucede porque `asyncio.create_task()` necesita un bucle de eventos en marcha. La solución es siempre colocar este tipo de código dentro de una función `async` y arrancarla con `asyncio.run()`.

Las tareas también se pueden cancelar o limitar en tiempo de ejecución:

In [9]:
import asyncio

async def proceso_largo():
    await asyncio.sleep(30)
    print("Proceso terminado")

async def main():
    tarea = asyncio.create_task(proceso_largo())
    tarea.cancel()  # cancelamos la tarea antes de que acabe

    try:
        await asyncio.wait_for(proceso_largo(), timeout=3.0)
    except asyncio.TimeoutError:
        print("El proceso superó el tiempo máximo permitido")

if __name__ == "__main__":
    asyncio.run(main())

El proceso superó el tiempo máximo permitido


## Medir el tiempo de ejecución con decoradores

Un decorador asíncrono te permite instrumentar tus corrutinas sin modificar su lógica interna. Es útil para detectar cuellos de botella:

In [10]:
import asyncio
import time

def cronometrar(func):
    async def envolver(*args, **kwargs):
        inicio = time.time()
        resultado = await func(*args, **kwargs)
        fin = time.time()
        print(f"'{func.__name__}' tardó {fin - inicio:.3f} segundos")
        return resultado
    return envolver

@cronometrar
async def consultar_base_datos():
    await asyncio.sleep(1.5)  # simula una consulta lenta
    print("Datos recuperados")

async def main():
    await consultar_base_datos()

if __name__ == "__main__":
    asyncio.run(main())

Datos recuperados
'consultar_base_datos' tardó 1.502 segundos


**¿Para qué sirve medir el tiempo en tareas asíncronas?**

Monitorizar el tiempo de ejecución tiene varias ventajas concretas:

Detectar cuellos de botella: una corrutina que tarda demasiado puede estar haciendo más trabajo del necesario o esperando recursos innecesariamente.

Planificar recursos: saber cuánto tarda cada tarea ayuda a decidir si conviene paralelizarla, delegarla a un worker externo o simplemente optimizarla.

Depurar comportamientos anómalos: si una tarea nunca termina o tarda mucho más de lo esperado, el tiempo de ejecución es la primera pista.

Mejorar la experiencia de usuario: en aplicaciones con interfaz, una tarea que bloquea por más de un par de segundos debería ir acompañada de algún indicador visual.

Validar la escalabilidad: antes de aumentar la carga de tu sistema, necesitas saber cómo se comportan tus corrutinas con los datos actuales.

## Manejo de errores en corrutinas concurrentes

Las corrutinas son "perezosas": no ejecutan su cuerpo hasta que alguien las espera con await. Esto significa que los errores también se posponen, lo que puede dificultar el diagnóstico si no estás preparado.

Cuando ejecutas múltiples corrutinas en paralelo, `asyncio.gather` con `return_exceptions=True` te permite recoger todos los errores sin que uno solo interrumpa el resto:

In [11]:
import asyncio

async def obtener_temperatura(ciudad):
    if ciudad == "Ciudad Desconocida":
        raise ValueError(f"No existe datos para: {ciudad}")
    await asyncio.sleep(0.5)
    return f"Temperatura en {ciudad}: 22°C"

async def obtener_cambio_divisa(par):
    if par == "ZZZ/XXX":
        raise KeyError(f"Par de divisas no válido: {par}")
    await asyncio.sleep(0.5)
    return f"Cambio {par}: 1.08"

async def main():
    resultados = await asyncio.gather(
        obtener_temperatura("Madrid"),
        obtener_temperatura("Ciudad Desconocida"),
        obtener_cambio_divisa("EUR/USD"),
        obtener_cambio_divisa("ZZZ/XXX"),
        return_exceptions=True
    )

    for r in resultados:
        if isinstance(r, Exception):
            print(f"Error capturado: {r}")
        else:
            print(r)

if __name__ == "__main__":
    asyncio.run(main())

Temperatura en Madrid: 22°C
Error capturado: No existe datos para: Ciudad Desconocida
Cambio EUR/USD: 1.08
Error capturado: 'Par de divisas no válido: ZZZ/XXX'


## El bucle de eventos

El bucle de eventos es el motor que orquesta la ejecución de todas las corrutinas. Normalmente no necesitas interactuar con él directamente —asyncio.run() lo gestiona— pero en algunos casos avanzados puedes controlarlo manualmente:

In [12]:
import asyncio

async def procesar_pedido(pedido_id):
    print(f"Pedido #{pedido_id}: iniciando procesamiento")
    await asyncio.sleep(pedido_id)  # simula tiempo variable por pedido
    print(f"Pedido #{pedido_id}: listo")

bucle = asyncio.get_event_loop()

tareas = [bucle.create_task(procesar_pedido(pid)) for pid in range(1, 4)]

bucle.run_until_complete(asyncio.gather(*tareas))

bucle.close()

Pedido #1: iniciando procesamiento
Pedido #2: iniciando procesamiento
Pedido #3: iniciando procesamiento
Pedido #1: listo
Pedido #2: listo
Pedido #3: listo


RuntimeError: Cannot close a running event loop

El problema anterior, es que en Jupyter Notebook ya hay un bucle de eventos corriendo, por lo que no puedes crear uno nuevo ni cerrarlo manualmente. La solución es usar directamente await con asyncio.gather():

In [13]:
import asyncio

async def procesar_pedido(pedido_id):
    print(f"Pedido #{pedido_id}: iniciando procesamiento")
    await asyncio.sleep(pedido_id)  # simula tiempo variable por pedido
    print(f"Pedido #{pedido_id}: listo")

async def main():
    tareas = [asyncio.create_task(procesar_pedido(pid)) for pid in range(1, 4)]
    await asyncio.gather(*tareas)

await main()

Pedido #1: iniciando procesamiento
Pedido #2: iniciando procesamiento
Pedido #3: iniciando procesamiento
Pedido #1: listo
Pedido #2: listo
Pedido #3: listo


Aquí lanzamos tres pedidos concurrentes con tiempos de procesamiento distintos (1, 2 y 3 segundos), y el bucle de eventos los coordina sin bloquear ninguno.

Recuerda: asyncio está pensado para tareas limitadas por entrada/salida (I/O-bound). Si tus operaciones consumen CPU intensivamente (cálculos matemáticos, procesamiento de imágenes), el paralelismo real requiere `multiprocessing`, no corrutinas.

## Modo debug: detecta problemas antes de que lleguen a producción

`asyncio` incluye un modo de depuración que emite advertencias cuando detecta patrones problemáticos, como llamadas bloqueantes dentro del bucle de eventos:

In [14]:
import os
os.environ['PYTHONASYNCIODEBUG'] = '1'
import logging
logging.basicConfig(level=logging.DEBUG)

La variable de entorno debe configurarse antes de importar asyncio. Una vez habilitado, el modo debug te avisará de problemas como este:

In [15]:
import os
os.environ['PYTHONASYNCIODEBUG'] = '1'
import asyncio
import time

async def enviar_notificacion(usuario_id):
    print(f"Enviando notificación a usuario {usuario_id}")
    await asyncio.sleep(1)
    print(f"Notificación enviada a usuario {usuario_id}")

async def exportar_csv():
    print("Exportando datos a CSV...")
    time.sleep(3)  # ⚠️ llamada bloqueante: no usar en código asíncrono
    print("CSV exportado")

async def main():
    tareas = [
        asyncio.create_task(enviar_notificacion(101)),
        asyncio.create_task(enviar_notificacion(202)),
        asyncio.create_task(exportar_csv())
    ]
    await asyncio.gather(*tareas)

bucle = asyncio.get_event_loop()
bucle.set_debug(True)

try:
    bucle.run_until_complete(main())
finally:
    bucle.close()

Enviando notificación a usuario 101
Enviando notificación a usuario 202
Exportando datos a CSV...
CSV exportado
Notificación enviada a usuario 101
Notificación enviada a usuario 202


RuntimeError: Cannot close a running event loop

Mismo problema que antes, la solución es la siguiente si se está trabajando con jupyter notebook

In [16]:
import os
os.environ['PYTHONASYNCIODEBUG'] = '1'
import asyncio
import time
import logging

logging.basicConfig(level=logging.DEBUG)

async def enviar_notificacion(usuario_id):
    print(f"Enviando notificación a usuario {usuario_id}")
    await asyncio.sleep(1)
    print(f"Notificación enviada a usuario {usuario_id}")

async def exportar_csv():
    print("Exportando datos a CSV...")
    time.sleep(3)  # ⚠️ llamada bloqueante — asyncio lo detectará en modo debug
    print("CSV exportado")

async def main():
    # Activar debug desde dentro del bucle ya activo
    asyncio.get_event_loop().set_debug(True)
    
    tareas = [
        asyncio.create_task(enviar_notificacion(101)),
        asyncio.create_task(enviar_notificacion(202)),
        asyncio.create_task(exportar_csv())
    ]
    await asyncio.gather(*tareas)

await main()

Enviando notificación a usuario 101
Enviando notificación a usuario 202
Exportando datos a CSV...
CSV exportado
Notificación enviada a usuario 101
Notificación enviada a usuario 202


En este ejemplo, `time.sleep(3)` dentro de `exportar_csv` bloquea el bucle de eventos durante 3 segundos, impidiendo que las otras tareas avancen. El modo debug detecta esta situación y la reporta con un mensaje de advertencia. En producción, la solución es reemplazar `time.sleep` por `await asyncio.sleep` o, si la operación es realmente síncrona, ejecutarla en un executor separado.

## Conclusión

Con este material se tiene una base sólida para trabajar con `asyncio`: sabes cómo definir y ejecutar corrutinas, cómo gestionar recursos de forma asíncrona con `async with` y `async for`, cómo crear tareas concurrentes, medir su rendimiento y capturar errores de forma robusta.

El verdadero potencial de `asyncio` aparece cuando las aplicaciones crecen y necesitas coordinar docenas o cientos de tareas simultáneas: semáforos para limitar concurrencia, colas para comunicación entre tareas, o transportes y protocolos para comunicación de red de bajo nivel. 